<a href="https://colab.research.google.com/github/fpellerano/devllm/blob/main/20_4_b_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To upload a `requirements.txt` file to your Google Colab environment, follow these steps:

1.  **Open the Files pane**: Click the **Folder icon** in the left-hand sidebar.
2.  **Upload your file**:
    *   **Drag and drop** your file directly into the pane.
    *   **OR** click the **Upload to session storage** icon (a file with an upward arrow) to select it from your local machine.

  
>[IMPORTANT] Files uploaded this way are temporary and will be deleted once the session is recycled.

# Chat Memory with LangGraph

## The Core Problem: LLMs Are Stateless

Every call to an LLM is completely independent. The model has no memory of previous
messages unless you explicitly include them in the current prompt.

This means **memory is not a feature of the model — it's a feature of your application**.
You decide what gets included in each call, and the different "memory strategies" are
just different policies for what to keep and what to discard.

Previously, LangChain offered classes like `ConversationBufferMemory` and
`ConversationSummaryMemory` to manage this. These have been **deprecated** in favor of
managing state explicitly through **LangGraph**, which gives you full control and visibility.

This notebook covers four strategies:

| Strategy | What gets passed to the model | Bounded? |
|---|---|---|
| **Full Buffer** | The entire conversation history | No — grows forever |
| **Window (K)** | Only the last K exchanges | Yes — fixed count |
| **Summary** | A running summary + recent messages | Yes — fixed size |
| **Token-Limited** | The most recent messages that fit a token budget | Yes — fixed tokens |

In [ ]:
!pip install -qU -r requirements.txt

In [ ]:
from google.colab import userdata

google_api_key = userdata.get("GOOGLE_API_KEY")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0.0,
    google_api_key=google_api_key,
)

## Thread IDs and Memory Isolation

LangGraph's `MemorySaver` checkpointer persists conversation state between calls.
Each conversation is identified by a **`thread_id`** — a string key you provide.

- Same `thread_id` → same conversation, history accumulates across calls
- Different `thread_id` → completely independent conversations, no shared state

This is how a single deployed agent can serve thousands of simultaneous users:
each user session gets a unique `thread_id` and their history never mixes.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict


def print_history(app, config, label="Message history"):
    """Inspect the messages currently stored in the graph's state."""
    state = app.get_state(config)
    msgs  = state.values.get("messages", [])
    print(f"{label}: {len(msgs)} message(s)")
    for msg in msgs:
        role    = type(msg).__name__.replace("Message", "")
        # Handle both string content and list-of-dicts content
        content = msg.content
        if isinstance(content, list):
            content = " ".join([part.get("text", "") if isinstance(part, dict) else str(part) for part in content])

        preview = content[:70].replace("\n", " ")
        ellipsis = "..." if len(content) > 70 else ""
        print(f"  [{role}] {preview}{ellipsis}")
    print()

## 1. Full Buffer Memory

The simplest strategy: **pass the entire conversation history** on every call.
The model always has full context — nothing is lost.

The cost: the number of tokens sent grows with every turn.
For long conversations, this becomes expensive and eventually exceeds the model's context window.

In [ ]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def call_model(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


workflow = StateGraph(state_schema=State)
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

buffer_app = workflow.compile(checkpointer=MemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "buffer_demo"}}

turns = [
    "Hi! My name is Marco and my favorite color is blue.",
    "What is the capital of Finland?",
    "Interesting! What is the oldest city in that country?",
    "What is 15 multiplied by 8?",
    "Do you remember my name and my favorite color?",   # ← memory stress test
]

print("=== Full Buffer Memory ===\n")
for turn in turns:
    result = buffer_app.invoke({"messages": [HumanMessage(turn)]}, config)
    print(f"Human : {turn}")

    # Handle list-based AI response content safely
    ai_content = result['messages'][-1].content
    if isinstance(ai_content, list):
        ai_content = ai_content[0].get('text', str(ai_content))

    print(f"AI    : {ai_content[:150]}\n")

print()
print_history(buffer_app, config, "Final state")

=== Full Buffer Memory ===

Human : Hi! My name is Marco and my favorite color is blue.
AI    : It’s nice to see you again, Marco! I definitely remember—your name is Marco and your favorite color is blue. 

Are we continuing our conversation abou

Human : What is the capital of Finland?
AI    : The capital of Finland is **Helsinki**.

Human : Interesting! What is the oldest city in that country?
AI    : The oldest city in Finland is **Turku**. 

As I mentioned earlier, it was founded in the 13th century and served as the capital of Finland for many ye

Human : What is 15 multiplied by 8?
AI    : 15 multiplied by 8 is **120**.

Human : Do you remember my name and my favorite color?
AI    : Yes, I certainly do! Your name is **Marco** and your favorite color is **blue**.


Final state: 20 message(s)
  [Human] Hi! My name is Marco and my favorite color is blue.
  [AI] Hi Marco! It’s nice to meet you. Blue is a fantastic choice—it’s such ...
  [Human] What is the capital of Finland?
  [AI] 

Notice the message count in the final state: every single exchange is stored.
The last question — "Do you remember my name?" — is answered correctly because
the very first message is still in the history.

With full buffer memory, **nothing is ever forgotten** — but the token cost
grows linearly with the conversation length.

## 2. Window Memory

Window memory keeps only the **last K exchanges** (K human + K AI messages).
Older messages are dropped from the context window before each model call.

The token cost is bounded — you always send at most `2K + 1` messages.
The tradeoff: anything said more than K turns ago is lost.

In [ ]:
def make_window_app(k: int):
    """Build a window memory app that keeps the last k exchanges."""

    def call_model_windowed(state: State):
        messages = state["messages"]
        # Keep the last k*2 messages (k human + k AI) plus the current one
        windowed = messages[-(k * 2 + 1):] if k > 0 else messages[-1:]
        response = llm.invoke(windowed)
        return {"messages": [response]}

    wf = StateGraph(state_schema=State)
    wf.add_edge(START, "model")
    wf.add_node("model", call_model_windowed)
    return wf.compile(checkpointer=MemorySaver())

In [ ]:
# Compare K=1 vs K=2 on the same 5-turn conversation
for k in [1, 2]:
    app_w = make_window_app(k)
    cfg   = {"configurable": {"thread_id": f"window_k{k}"}}

    print(f"=== Window Memory K={k} ===\n")
    for turn in turns:
        result = app_w.invoke({"messages": [HumanMessage(turn)]}, cfg)
        print(f"Human : {turn}")

        # Extract content safely for display
        ai_content = result['messages'][-1].content
        if isinstance(ai_content, list):
            ai_content = ai_content[0].get('text', str(ai_content))

        print(f"AI    : {ai_content[:150]}\n")
    print("-" * 60 + "\n")

=== Window Memory K=1 ===

Human : Hi! My name is Marco and my favorite color is blue.
AI    : [{'type': 'text', 'text': 'Hi Marco! It’s nice to meet you. Blue is a fantastic choice—it’s such a calming and versatile color. Whether it’s the deep blue of the ocean or the bright blue of a clear sky, it’s always a classic.\n\nHow is your day going so far?', 'extras': {'signature': 'EjQKMgG+Pvb7ruuD3rNH5wW65qdtiwrhZzTYWlcd7xJYjJjBrapuKYG8a2My6mCoZMe4hYOm'}}]

Human : What is the capital of Finland?
AI    : [{'type': 'text', 'text': 'The capital of Finland is **Helsinki**.', 'extras': {'signature': 'EjQKMgG+Pvb7MdJK3vye48o3/jBEJNUXug6nS6u4WWxnm4HmXyzp5dY8nlXthhIarYplABSG'}}]

Human : Interesting! What is the oldest city in that country?
AI    : [{'type': 'text', 'text': "The oldest city in Finland is **Turku**.\n\nIt was founded in the 13th century (traditionally cited as 1229) and served as the country's capital until 1812, when the capital was moved to Helsinki by the Russian Empire. Becau

With `K=1`, the model only sees the immediately preceding exchange. By turn 5,
it has no idea who "Marco" is or what his favorite color is.

With `K=2`, it can resolve references to the previous topic ("that country") but
still loses facts from further back.

**The window size is a direct tradeoff between cost and how far back the model can remember.**

## 3. Summary Memory

Summary memory takes a different approach: instead of keeping raw messages,
it **compresses the conversation history into a running summary** after each turn.

Each new turn sees: the current summary + the most recent messages.
Old messages are replaced by the summary, keeping the token count bounded
while (ideally) preserving the important facts.

The summary is updated after every exchange, so important details mentioned early
in a long conversation are not lost — they're distilled into the summary.

In [ ]:
from langgraph.graph import END


class SummaryState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    summary: str


def call_model_with_summary(state: SummaryState):
    # Prepend the running summary as a system message (if one exists)
    if state.get("summary"):
        sys_msg  = SystemMessage(f"Summary of the conversation so far:\n{state['summary']}")
        messages = [sys_msg] + state["messages"]
    else:
        messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


def summarize_conversation(state: SummaryState):
    # Build the summarization prompt
    if state.get("summary"):
        prompt = (
            f"Existing summary:\n{state['summary']}\n\n"
            "Update this summary to include the new messages above. "
            "Keep it concise but preserve all specific facts and names."
        )
    else:
        prompt = (
            "Summarize the conversation above into a concise paragraph. "
            "Preserve all specific facts, names, and numbers."
        )

    messages  = state["messages"] + [HumanMessage(prompt)]
    new_summary = llm.invoke(messages).content

    # Only keep the 2 most recent messages — the summary replaces the rest
    return {"summary": new_summary, "messages": state["messages"][-2:]}


wf_summary = StateGraph(state_schema=SummaryState)
wf_summary.add_node("model",     call_model_with_summary)
wf_summary.add_node("summarize", summarize_conversation)
wf_summary.add_edge(START,       "model")
wf_summary.add_edge("model",     "summarize")
wf_summary.add_edge("summarize", END)

summary_app = wf_summary.compile(checkpointer=MemorySaver())

In [ ]:
cfg_summary = {"configurable": {"thread_id": "summary_demo"}}

print("=== Summary Memory ===\n")
for turn in turns:
    result = summary_app.invoke({"messages": [HumanMessage(turn)]}, cfg_summary)
    print(f"Human   : {turn}")
    print(f"AI      : {result['messages'][-1].content[:150]}")

    # Show the evolving summary after each turn
    state = summary_app.get_state(cfg_summary)
    summary = state.values.get("summary", "")
    if summary:
        print(f"Summary : {summary[:120]}...")
    print()

=== Summary Memory ===

Human   : Hi! My name is Marco and my favorite color is blue.
AI      : [{'type': 'text', 'text': 'Hi Marco! It’s nice to meet you. Blue is a fantastic choice—it’s such a calming and versatile color. Whether it’s the deep blue of the ocean or the bright blue of a clear sky, it’s always a classic.\n\nHow is your day going so far?', 'extras': {'signature': 'EjQKMgG+Pvb7isObNeHeqT2RSFiW+2FI6jFK1iwuj1vjrwxQRxULQPWMcOKzph3i/kz3U2H6'}}]
Summary : [{'type': 'text', 'text': 'Marco introduced himself and shared that his favorite color is blue.', 'extras': {'signature': 'EjQKMgG+Pvb7E9DoSn6uwS9Y0OTPbJSQ+oXGFSlcT/xCK6TJSShCsICXhGQKvUYfkK+Ym+Z3'}}]...

Human   : What is the capital of Finland?
AI      : [{'type': 'text', 'text': 'The capital of Finland is **Helsinki**.', 'extras': {'signature': 'EjQKMgG+Pvb7e7VXisz5GiNDRzyWP0kKlg0R5s1PLdaH84QOVXpPpo1TSPoeiI+CDFqTm2vi'}}]
Summary : [{'type': 'text', 'text': "[{'type': 'text', 'text': 'Marco introduced himself, shared that hi

Notice how the summary evolves with each turn, accumulating key facts.
By turn 5, the summary contains "Marco" and "blue" even though those raw messages
were long since replaced — the model can still answer the memory stress test correctly.

**Summary memory trades raw message fidelity for compact long-term retention.**
The quality of the summarization model directly affects what gets preserved.

## 4. Token-Limited Memory

Token-limited memory trims the message list to fit within a fixed **token budget**,
always keeping the most recent messages. Older messages are dropped when the total
exceeds `MAX_TOKENS`.

This gives precise control over cost — you know exactly how much context you're sending —
and degrades gracefully: as the conversation grows, older turns are silently dropped.

LangChain's `trim_messages` handles the counting and trimming using the LLM's own tokenizer.

In [ ]:
from langchain_core.messages import trim_messages

MAX_TOKENS = 250   # intentionally tight to show trimming behavior


def call_model_token_limited(state: State):
    trimmed = trim_messages(
        state["messages"],
        max_tokens=MAX_TOKENS,
        strategy="last",          # keep the most recent messages
        token_counter=llm,        # use the model's own token counter
        allow_partial=False,      # never split a message mid-way
        include_system=True,
    )
    # Show how many messages survived trimming
    print(f"  [trim: {len(state['messages'])} → {len(trimmed)} messages sent to model]")
    response = llm.invoke(trimmed)
    return {"messages": [response]}


wf_tokens = StateGraph(state_schema=State)
wf_tokens.add_edge(START, "model")
wf_tokens.add_node("model", call_model_token_limited)

token_app = wf_tokens.compile(checkpointer=MemorySaver())

In [ ]:
cfg_tokens = {"configurable": {"thread_id": "tokens_demo"}}

print(f"=== Token-Limited Memory (MAX_TOKENS={MAX_TOKENS}) ===\n")
for turn in turns:
    result = token_app.invoke({"messages": [HumanMessage(turn)]}, cfg_tokens)
    print(f"Human : {turn}")
    print(f"AI    : {result['messages'][-1].content[:150]}\n")

=== Token-Limited Memory (MAX_TOKENS=250) ===

  [trim: 1 → 1 messages sent to model]
Human : Hi! My name is Marco and my favorite color is blue.
AI    : [{'type': 'text', 'text': 'Hi Marco! It’s nice to meet you. Blue is a fantastic choice—it’s such a calming and versatile color. Whether it’s the deep blue of the ocean or the bright blue of a clear sky, it’s always a classic.\n\nHow is your day going so far?', 'extras': {'signature': 'EjQKMgG+Pvb746pfWb786fcCfPP1EfJd90bjAAudwsvxK9/5MScs6l/BpYRPMaGFfD/BFR0t'}}]

  [trim: 3 → 3 messages sent to model]
Human : What is the capital of Finland?
AI    : [{'type': 'text', 'text': 'The capital of Finland is **Helsinki**.', 'extras': {'signature': 'EjQKMgG+Pvb7mVuqVtjjw5KVayZsf0b9Q6fgl2mS9NwMdTK66ZZdiMr2KzTNdehDjbzUJPsm'}}]

  [trim: 5 → 5 messages sent to model]
Human : Interesting! What is the oldest city in that country?
AI    : [{'type': 'text', 'text': "The oldest city in Finland is **Turku**.\n\nIt was founded in the 13th century (around 

Watch the `[trim: X → Y messages]` line as the conversation progresses.
Once the history exceeds the budget, early messages are silently dropped.
Whether the final memory question is answered correctly depends on whether
the first exchange survived the trimming.

## Comparing All Strategies

Let's run the same memory stress test — a 5-turn conversation where a key fact
is introduced in turn 1 and tested in turn 5 — through all four strategies at once.

In [ ]:
stress_test = [
    "My name is Sofia and I am allergic to peanuts.",
    "What is the boiling point of water at sea level?",
    "Who wrote the novel '1984'?",
    "What year did the Berlin Wall fall?",
    "What is my name, and is there anything I should avoid eating?",  # ← the test
]

strategies = {
    "Buffer (full)":      (buffer_app,  {"configurable": {"thread_id": "stress_buffer"}}),
    "Window K=1":         (make_window_app(1), {"configurable": {"thread_id": "stress_window1"}}),
    "Window K=2":         (make_window_app(2), {"configurable": {"thread_id": "stress_window2"}}),
    "Summary":            (summary_app, {"configurable": {"thread_id": "stress_summary"}}),
    "Token-limit 250":    (token_app,   {"configurable": {"thread_id": "stress_tokens"}}),
}

final_answers = {}

for name, (app, cfg) in strategies.items():
    for turn in stress_test:
        result = app.invoke({"messages": [HumanMessage(turn)]}, cfg)
    final_answers[name] = result["messages"][-1].content

print("=== Final answer to: 'What is my name, and is there anything I should avoid eating?' ===\n")
for name, answer in final_answers.items():
    print(f"[{name}]")
    print(f"  {answer[:200]}")
    print()

  [trim: 1 → 1 messages sent to model]
  [trim: 3 → 3 messages sent to model]


## When to Use Each Strategy

| Strategy | Token cost | Long-term recall | Extra LLM calls | Best for |
|---|---|---|---|---|
| **Full Buffer** | Grows unboundedly | Perfect | None | Short conversations, prototypes |
| **Window (K)** | Fixed: ~2K messages | Only last K turns | None | Task-focused, stateless interactions |
| **Summary** | Small + recent msgs | Good (summary quality depends on LLM) | 1 per turn | Long-running conversations |
| **Token-Limited** | Fixed token budget | Recent messages only | None | Cost-sensitive, predictable billing |

### A Note on Production Memory

`MemorySaver` stores state **in memory** — it is lost when the process restarts.
For production deployments, replace it with a persistent checkpointer:

```python
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.postgres import PostgresSaver

memory = SqliteSaver.from_conn_string("conversations.db")
```

The rest of your code stays the same — only the checkpointer changes.

## References

- [LangGraph: Memory](https://langchain-ai.github.io/langgraph/concepts/memory/)
- [LangGraph: Persistence & Checkpointers](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [LangChain: `trim_messages`](https://python.langchain.com/docs/how_to/trim_messages/)
- [LangChain: How to add memory to chatbots](https://python.langchain.com/docs/how_to/chatbots_memory/)
- [LangGraph: Build a Chatbot with Memory](https://langchain-ai.github.io/langgraph/tutorials/introduction/)